# Семинар 12. PyTorch по одной операции

Тензоры, данные, модель и обучение: запускаем маленькую ячейку и рассматриваем результат.
Для примеров нужны NumPy, PyTorch, Matplotlib, ipywidgets и tqdm. Данные помещаются в несколько строк; скачивать датасет не требуется.

Для самостоятельного разбора: [PyTorch: Learn the Basics](https://docs.pytorch.org/tutorials/beginner/basics/intro.html), [Dive into Deep Learning](https://en.d2l.ai/chapter_preface/index.html).

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

In [ ]:
torch.__version__

## 1. Тензор: значения, форма, тип и устройство

`torch.Tensor` хранит многомерный массив чисел. Скаляр имеет ноль осей, вектор - одну, матрица - две. Число осей и количество элементов - разные характеристики.

Создадим скаляр. Пустая форма `()` означает отсутствие осей.

In [ ]:
scalar = torch.tensor(3.)
scalar, scalar.shape

Вектор из трёх целых чисел.

In [ ]:
vector = torch.tensor([1, 2, 3])
vector

Матрица: две строки, три столбца. Явно зададим `float32`.

In [ ]:
matrix = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
matrix

Форма, число осей и количество чисел.

In [ ]:
matrix.shape, matrix.ndim, matrix.numel()

Тип чисел и место хранения.

In [ ]:
matrix.dtype, matrix.device

Последовательность чисел и новая форма. Число элементов сохраняется.

In [ ]:
torch.arange(6).reshape(2, 3)

Нули заданной формы.

In [ ]:
torch.zeros(2, 3)

Единицы с формой, типом и устройством другого тензора.

In [ ]:
torch.ones_like(matrix)

Нормальные случайные числа. Seed позволяет повторить последовательность в том же окружении.

In [ ]:
torch.manual_seed(42)
torch.randn(2, 3)

Десятичная точка меняет тип по умолчанию: `[1, 2]` даёт целые числа, `[1., 2.]` - вещественные. Для входов сети обычно нужен `float32`, для номеров классов в `CrossEntropyLoss` - `int64` (`torch.long`).

### Методы тензора, к которым будем возвращаться

| Нужно | Выражение | Что получится |
|:--|:--|:--|
| Проверить форму и тип | `x.shape`, `x.ndim`, `x.dtype`, `x.device` | Свойства тензора |
| Посчитать элементы | `x.numel()` | Количество элементов |
| Изменить форму | `x.reshape(...)`, `x.flatten(start_dim=1)` | Представление или копия |
| Переставить оси | `x.permute(...)`, `x.transpose(0, 1)` | Представление с другими шагами в памяти |
| Добавить или убрать ось длины 1 | `x.unsqueeze(dim)`, `x.squeeze(dim)` | Представление с другой формой |
| Свести ось к числу | `x.sum(dim=...)`, `x.mean(dim=...)` | Сумма или среднее вдоль оси |
| Выбрать класс | `x.argmax(dim=1)` | Индекс максимума в каждой строке |
| Указать устройство и тип | `x.to(device=..., dtype=...)` | Тензор на нужном устройстве, при совпадении условий обычно сам `x` |
| Скопировать данные | `x.clone()` | Новая память и связь с графом |
| Остановить градиент | `x.detach()` | Тензор с общей памятью, отсоединённый от графа |
| Прочитать одно число в Python | `x.item()` | Python-скаляр из тензора с одним элементом |

## 2. NumPy и PyTorch: похожий интерфейс, общая память

| NumPy | PyTorch | Смысл |
|:--|:--|:--|
| `np.array(data)` | `torch.tensor(data)` | создание массива с копированием данных |
| `array.shape` | `tensor.shape` | форма |
| `array.astype(np.float32)` | `tensor.float()` | преобразование типа |
| `array.reshape(...)` | `tensor.reshape(...)` | изменение формы |
| `array.mean(axis=0)` | `tensor.mean(dim=0)` | усреднение по оси |
| `array @ other` | `tensor @ other` | матричное произведение |

`torch.from_numpy` для поддерживаемого NumPy-массива на CPU разделяет с ним память. `torch.tensor` создаёт копию. Проверим на изменении одного числа.

In [ ]:
array = np.array([1., 2., 3.], dtype=np.float32)
array

In [ ]:
shared = torch.from_numpy(array)
copied = torch.tensor(array)

Изменим исходный массив и посмотрим оба тензора.

In [ ]:
array[0] = 100
shared, copied

Обратное преобразование CPU-тензора также может разделять память. Для независимой копии нужна `.copy()`.

In [ ]:
shared.numpy()

`clone()` создаёт отдельные данные. В отличие от него, простое присваивание даёт второе имя того же объекта.

In [ ]:
independent = shared.clone()
independent[1] = -10
shared, independent

## 3. Индексы, операции и оси

Индексы начинаются с нуля. В записи `matrix[row, column]` сначала выбирают строку, затем столбец.

Второй элемент первой строки.

In [ ]:
matrix[0, 1]

Весь второй столбец.

In [ ]:
matrix[:, 1]

Маска и отбор элементов.

In [ ]:
matrix > 3, matrix[matrix > 3]

Поэлементное умножение: форма сохраняется.

In [ ]:
matrix * matrix

Матричное умножение: `(2, 3) @ (3, 2)` даёт `(2, 2)`.

In [ ]:
matrix @ matrix.T

**Broadcasting в PyTorch работает по тем же правилам, что и в NumPy.** Вектор длины 3 прибавляется к каждой строке.

In [ ]:
matrix + torch.tensor([10., 20., 30.])

In [ ]:
np.array_equal((matrix + torch.tensor([10., 20., 30.])).numpy(), matrix.numpy() + [10, 20, 30])

При broadcasting формы сравнивают справа налево. Размеры должны совпадать или один из них должен быть равен 1. В примере `(2, 3)` и `(3,)` совместимы: вектор используется в обеих строках. Это правило такое же, как в NumPy.

`dim=0`: усредним строки. Останется одно число для каждого столбца.

In [ ]:
matrix.mean(dim=0)

`dim=1`: усредним столбцы. Останется одно число для каждой строки.

In [ ]:
matrix.mean(dim=1)

`keepdim=True` сохраняет усреднённую ось длины 1. Получается форма `(2, 1)`, удобная для broadcasting.

In [ ]:
row_mean = matrix.mean(dim=1, keepdim=True)
row_mean, matrix - row_mean

Максимум каждой строки: отдельно значения и их индексы.

In [ ]:
matrix.max(dim=1)

`cat` продолжает существующую ось, `stack` добавляет новую.

In [ ]:
torch.cat([matrix, matrix], dim=0).shape, torch.stack([matrix, matrix], dim=0).shape

Обычный срез тензора, как и базовый срез NumPy, разделяет данные с исходным массивом. Изменение среза может изменить исходный тензор; независимую копию создают через `clone()`. Индексация маской выше создаёт отдельный результат.

## 4. Из картинки HWC в батч NCHW

В NumPy цветная картинка часто имеет форму `H x W x C`: высота, ширина, каналы. `Conv2d` принимает батч `N x C x H x W`. Ось N - число изображений. Оси переставляют через `permute`; `reshape` только переупаковывает последовательность элементов и не заменяет перестановку осей.

Маленькая «картинка»: 2 строки, 2 столбца, 3 канала. Видно каждое число.

In [ ]:
image_hwc = torch.arange(12).reshape(2, 2, 3)
image_hwc

Поставим ось каналов первой.

In [ ]:
image_chw = image_hwc.permute(2, 0, 1)
image_chw

Сравним с `reshape`: одинаковая форма не означает одинаковые значения на позициях.

In [ ]:
torch.equal(image_chw, image_hwc.reshape(3, 2, 2))

Переведём числа в `float32`, приведём значения к диапазону [0, 1] и добавим ось батча.

In [ ]:
image_batch = (image_chw.float() / 11).unsqueeze(0)
image_batch.shape

`flatten(start_dim=1)` сохраняет батч и объединяет все остальные оси.

In [ ]:
image_batch.flatten(start_dim=1)

Для обычной картинки `uint8` делят на 255. В нашем примере максимальное число 11.

### Форма и расположение в памяти

`stride()` показывает, на сколько **элементов памяти** надо сдвинуться, чтобы увеличить индекс вдоль оси на единицу. У HWC соседние каналы лежат рядом, поэтому по C шаг 1. После `permute` форма становится CHW, но данные лежат по прежним адресам. Теперь шаг 1 стоит в начале списка, рядом с осью C.

In [ ]:
image_hwc.stride(), image_chw.stride()

In [ ]:
image_chw.is_contiguous(), image_chw.contiguous().is_contiguous()

`is_contiguous()` проверяет, идут ли элементы подряд в порядке осей тензора. `contiguous()` укладывает их в таком порядке. Для уже непрерывного тензора вызов возвращает тот же объект. Копия нужна, например, перед `view(-1)`, если текущие шаги не позволяют объединить оси. `reshape(-1)` сам выберет представление или копию.

In [ ]:
packed_chw = image_chw.contiguous()
packed_chw.stride(), torch.equal(packed_chw, image_chw)

In [ ]:
image_chw.data_ptr() == image_hwc.data_ptr(), packed_chw.data_ptr() == image_chw.data_ptr()

Сравним одну и ту же сумму по строкам на CPU. Сначала берём транспонированное представление, затем заранее уложенную копию. В третьем замере оплачиваем копирование каждый раз. Скорость зависит от операции, размеров и устройства. Если копия нужна для нескольких операций, её стоимость можно распределить между ними.

In [ ]:
from torch.utils.benchmark import Timer

strided_matrix = torch.rand(1024, 1024).T
packed_matrix = strided_matrix.contiguous()
torch.allclose(strided_matrix.sum(dim=1), packed_matrix.sum(dim=1))

In [ ]:
expressions = [
    "strided_matrix.sum(dim=1)",
    "packed_matrix.sum(dim=1)",
    "strided_matrix.contiguous().sum(dim=1)",
]
for expression in expressions:
    timer = Timer(expression, globals=globals(), num_threads=1)
    measurement = timer.blocked_autorange(min_run_time=0.15)
    print(f"{measurement.median * 1e6:7.1f} мкс  {expression}")

## 5. CPU, CUDA и MPS

CPU - центральный процессор; CUDA - интерфейс вычислений на поддерживаемых GPU NVIDIA; MPS (Metal Performance Shaders) - backend PyTorch для поддерживаемых устройств Apple. Доступность зависит от железа и установленной сборки.

In [ ]:
available = {
    'cpu': True,
    'cuda': torch.cuda.is_available(),
    'mps': torch.backends.mps.is_available(),
}
available

Выберем доступный ускоритель; без него примеры выполнятся на CPU.

In [ ]:
if available['cuda']:
    device = torch.device('cuda')
elif available['mps']:
    device = torch.device('mps')
else:
    device = torch.device('cpu')
device

`.to(device)` возвращает тензор на нужном устройстве. Исходный тензор сам не переносится.

In [ ]:
on_device = matrix.to(device)
matrix.device, on_device.device

Вернём данные на CPU и преобразуем в NumPy.

In [ ]:
on_device.cpu().numpy()

Модель и её входы должны находиться на согласованном устройстве. То же относится к меткам при вычислении loss. Модель переносят до создания оптимизатора. На маленьких массивах ускоритель не обязан быть быстрее: перенос данных и запуск операций тоже стоят времени.

### Одно число на CPU через item

`loss.item()` возвращает Python-число для печати или логирования. Тензор должен содержать ровно один элемент. На CUDA операции обычно ставятся в очередь, а Python продолжает работу. Когда Python просит число, ему приходится дождаться вычислений, от которых оно зависит. Вызов на каждом батче может прерывать эту работу в очереди и снижать пропускную способность. Для лога можно собрать несколько `loss.detach()` на устройстве и прочитать среднее одним `item()` раз в несколько шагов или за эпоху.

[PyTorch о выполнении операций CUDA](https://docs.pytorch.org/docs/stable/notes/cuda.html#asynchronous-execution).

In [ ]:
mean_on_device = on_device.mean()
mean_on_device.item(), type(mean_on_device.item())

## 6. Dataset: как получить один объект

`Dataset` описывает доступ к примерам. Метод `__len__` отвечает на `len(dataset)`, метод `__getitem__` - на `dataset[index]`. В задаче с учителем один пример - пара `(признаки, метка)`.

Создадим шесть двумерных точек двух классов.

In [ ]:
features = torch.tensor([[-2., -1.], [-1., -2.], [-1., -1.], [1., 1.], [1., 2.], [2., 1.]])
targets = torch.tensor([0, 0, 0, 1, 1, 1])
features, targets

У датасета с доступом по индексу переопределяем `__getitem__(index)`. Для обычного `DataLoader` с выборкой по индексам задаём также `__len__()`, чтобы sampler знал число объектов. В `__init__` храним данные или список путей. При чтении картинок `__getitem__` может открыть файл и применить преобразования. Опциональный `__getitems__(indices)` позволяет получить список объектов одним вызовом. Потоковый `IterableDataset` вместо индекса реализует `__iter__()`. С несколькими воркерами поток разделяют между ними, чтобы объекты не дублировались.

In [ ]:
class PointsDataset(Dataset):
    def __init__(self, features, targets):
        self.features = features
        self.targets = targets

    def __len__(self):
        size = len(self.targets)
        return size

    def __getitem__(self, index):
        example = self.features[index], self.targets[index]
        return example

In [ ]:
dataset = PointsDataset(features, targets)
len(dataset), dataset[0]

Если данные уже лежат в тензорах и преобразований нет, готовый `TensorDataset` делает то же самое.

In [ ]:
TensorDataset(features, targets)[0]

## 7. DataLoader: как получить батч

`DataLoader` берёт объекты из Dataset, при необходимости перемешивает индексы и объединяет примеры в батчи. По умолчанию для тензоров применяется объединение `stack`: две точки формы `(2,)` превращаются в батч `(2, 2)`.

`num_workers=0` означает загрузку данных в основном процессе. Это простой вариант для ноутбука; отдельные процессы нужны, когда подготовка данных становится узким местом.

In [ ]:
loader = DataLoader(dataset, batch_size=2, shuffle=False, num_workers=0)
len(dataset), len(loader)

`iter(loader)` создаёт итератор, `next` получает следующий батч.

In [ ]:
batches = iter(loader)
batch_features, batch_targets = next(batches)
batch_features, batch_targets

Первый размер обеих частей - число объектов в батче.

In [ ]:
batch_features.shape, batch_targets.shape

In [ ]:
from support.pytorch_visuals import plot_batches
plot_batches(features, targets, batch_size=2);

Если размер Dataset не делится на размер батча, последний батч меньше. `drop_last=True` исключил бы его.

In [ ]:
short_loader = DataLoader(TensorDataset(features[:5], targets[:5]), batch_size=2)
[x.shape for x, y in short_loader]

На обучении обычно используют `shuffle=True`, на оценке - `False`. Для воспроизводимого порядка передают `generator=torch.Generator().manual_seed(42)`. Перемешивание не разрывает соответствие между объектом и меткой.

### Что настраивает DataLoader

| Аргумент | Что управляет |
|:--|:--|
| `batch_size` | Сколько объектов передаём модели за раз |
| `shuffle` | Перемешивание индексов перед эпохой |
| `sampler` | Свой порядок или вероятности выбора индексов. Используется без `shuffle=True` |
| `batch_sampler` | Готовые списки индексов батчей. Заменяет `batch_size`, `shuffle`, `sampler`, `drop_last` |
| `drop_last` | Отбрасывает неполный батч |
| `collate_fn` | Объединяет список объектов в батч |
| `num_workers` | Число отдельных процессов для чтения, преобразований и сборки. При 0 всё делает основной процесс |
| `prefetch_factor` | Сколько батчей заранее готовит каждый воркер. Обычно 2 при `num_workers > 0` |
| `persistent_workers` | Оставлять ли воркеры живыми между эпохами. Требует `num_workers > 0` |
| `pin_memory` | Помещает CPU-тензоры батча в закреплённую память для переноса на CUDA |
| `generator`, `worker_init_fn` | Генератор индексов и начальных seed воркеров, функция начальной настройки воркера |
| `timeout` | Сколько секунд ждать батч от воркеров. При 0 ограничение отключено |
| `in_order` | При `False` разрешает вернуть готовый батч раньше предыдущего. Используется с воркерами |

### Собираем свой батч

Стандартный сборщик складывает тензоры одинаковых форм вдоль новой оси. Свой сборщик полезен, если нужно дополнить изображения или последовательности до общей длины, вернуть маску или собрать словарь. Он получает список результатов `Dataset.__getitem__`. Здесь возвращаем две плотные группы тензоров.

In [ ]:
def collate_points(examples):
    points, labels = zip(*examples)
    batch = {"points": torch.stack(points), "labels": torch.stack(labels)}
    return batch

In [ ]:
dictionary_loader = DataLoader(dataset, batch_size=2, collate_fn=collate_points)
next(iter(dictionary_loader))

### Как подавать данные на GPU

Воркеры читают файлы, декодируют и собирают батчи на CPU. Основной процесс переносит готовый батч и запускает модель. При использовании CUDA `pin_memory=True` готовит закреплённую память на **хосте**, а `.to(device, non_blocking=True)` позволяет Python продолжить работу до окончания копирования. Перекрытие копирования и вычислений на GPU требует отдельной организации потоков CUDA и зависит от оборудования.

- Начинайте с `num_workers=0` и измеряйте время эпохи. Для медленного декодирования попробуйте 2 и 4. Если данных мало или они уже в памяти, запуск процессов может оказаться дороже полезной работы.
- `batch_size` влияет на память устройства, занятую входами и активациями. `num_workers` и `prefetch_factor` влияют прежде всего на память CPU. При 4 воркерах и prefetch 2 в подготовке могут быть 8 батчей. Если батч занимает 16 МиБ, это порядка 128 МиБ плюс память преобразований, копий и текущего батча.
- При росте `batch_size` каждый батч тяжелее и для CPU, и для GPU. `persistent_workers=True` экономит повторный запуск процессов, сохраняя их в памяти между эпохами.
- В скрипте на macOS и Windows функции датасета и сборщика должны быть доступны для импорта, а запуск защищают `if __name__ == "__main__":`. Здесь используем стандартный `TensorDataset` и импортируем сборщик из `support`.

[Аргументы DataLoader](https://docs.pytorch.org/docs/stable/data.html). [Примеры настройки загрузки](https://docs.pytorch.org/tutorials/intermediate/intermediate_data_loading_tutorial.html). [Закреплённая память и non_blocking](https://docs.pytorch.org/tutorials/intermediate/pinmem_nonblock.html).

In [ ]:
from support.loader_examples import collate_points as worker_collate

worker_loader = DataLoader(
    TensorDataset(features, targets), batch_size=2,
    num_workers=2, prefetch_factor=2, persistent_workers=True,
    collate_fn=worker_collate, pin_memory=device.type == "cuda",
)

In [ ]:
for cpu_batch in worker_loader:
    device_points = cpu_batch["points"].to(device, non_blocking=device.type == "cuda")
    device_labels = cpu_batch["labels"].to(device, non_blocking=device.type == "cuda")
    print(device_points.shape, device_labels.device)

In [ ]:
del worker_loader

## 8. Module: слой и своя модель

`nn.Module` - базовый класс слоёв и моделей. В `__init__` создают слои, в `forward` описывают вычисление выхода. `super().__init__()` инициализирует механизм регистрации параметров и вложенных модулей.

`nn.Linear(2, 2)` принимает два признака и возвращает две оценки классов. Веса имеют форму `(out_features, in_features)`: для батча строк вычисляют `Y = X @ W.T + b`.

`torch.nn` содержит базовый класс `Module`, готовые слои, функции потерь, контейнеры `Sequential` и `ModuleList`, параметры `Parameter`. В `nn.functional` доступны операции в виде функций. Например, `nn.Linear` хранит обучаемые веса и смещение, а `nn.functional.linear` получает их аргументами. [Документация torch.nn](https://docs.pytorch.org/docs/stable/nn.html).

В своём `Module` определяем `forward`. Если у слоя есть состояние, задаём его в `__init__` после `super().__init__()`. `nn.Parameter` регистрирует обучаемое значение, `register_buffer` подходит для состояния без градиента. Зарегистрированные параметры и буферы перемещаются вместе с моделью через `.to(device)`. PyTorch строит обратный проход из операций внутри `forward`: сложение, умножение и готовые слои уже знают свои локальные производные. Если пишем собственную дифференцируемую операцию с ручным backward, используем `torch.autograd.Function`.

In [ ]:
class TinyClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 2)

    def forward(self, values):
        logits = self.linear(values)
        return logits

In [ ]:
torch.manual_seed(42)
model = TinyClassifier().to(device)
model

Вызов `model(...)` запускает `forward` через механизм Module. Обычно напрямую `forward` не вызывают.

In [ ]:
model(features.to(device))

Параметры зарегистрированы автоматически. Оптимизатор получит их через `model.parameters()`.

In [ ]:
[(name, tuple(parameter.shape), parameter.requires_grad) for name, parameter in model.named_parameters()]

### Несколько слоёв через Sequential

`nn.Sequential` передаёт результат каждого слоя следующему. На маленькой картинке из раздела 4 посмотрим те же слои, что используются в CNN: свёртка, ReLU, pooling, Flatten и Linear.

Формы: `1 x 3 x 2 x 2` -> `1 x 2 x 2 x 2` -> `1 x 2 x 1 x 1` -> `1 x 2` -> `1 x 2`. ReLU форму не меняет.

In [ ]:
image_model = nn.Sequential(
    nn.Conv2d(3, 2, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(2, 2),
)
image_model(image_batch)

В этой модели `MaxPool2d(2)` превращает карту 2x2 в 1x1. `AvgPool2d(2)` взял бы среднее вместо максимума. `AdaptiveAvgPool2d((1, 1))` задал бы целевую форму выхода 1x1, независимо от исходных пространственных размеров.

### Выбираем функцию потерь

| Задача | Потеря | Что подать |
|:--|:--|:--|
| Предсказать число, сильнее штрафовать большие ошибки | `nn.MSELoss` | Предсказание и вещественный ответ одинаковой формы |
| Предсказать число, сделать штраф линейным по ошибке | `nn.L1Loss` | Такие же тензоры |
| Регрессия с квадратичным штрафом около нуля и линейным вдали | `nn.SmoothL1Loss` | Такие же тензоры |
| Ровно один класс из C | `nn.CrossEntropyLoss` | Сырые оценки `(N, C)` и номера классов `(N,)` типа `long` |
| Бинарный ответ или несколько независимых меток | `nn.BCEWithLogitsLoss` | Сырые оценки и вещественные метки 0/1 той же формы |
| Один класс, модель уже возвращает логарифмы вероятностей | `nn.NLLLoss` | Логарифмы вероятностей и номера классов |

`CrossEntropyLoss` сам вычисляет `log_softmax`. `BCEWithLogitsLoss` объединяет сигмоиду и бинарную кросс-энтропию. В этих примерах передаём сырые оценки. `reduction="none"` оставляет отдельные потери, `"sum"` их складывает, `"mean"` усредняет по правилам конкретной функции.

In [ ]:
regression = torch.tensor([2., 5.])
answer = torch.tensor([1., 3.])
nn.MSELoss(reduction="none")(regression, answer), nn.L1Loss(reduction="none")(regression, answer)

In [ ]:
class_scores = torch.tensor([[2., 0., -1.], [-1., 0., 2.]])
nn.CrossEntropyLoss(reduction="none")(class_scores, torch.tensor([0, 2]))

In [ ]:
nn.BCEWithLogitsLoss(reduction="none")(torch.tensor([2., -2.]), torch.tensor([1., 0.]))

## 9. Один шаг обучения по строкам

Перенесём на устройство все шесть точек и метки. Для этого маленького примера используем один батч. `CrossEntropyLoss` принимает сырые оценки классов, без предварительного `softmax`, и целочисленные номера правильных классов.

In [ ]:
train_x = features.to(device)
train_y = targets.to(device)
train_x.device, train_y.dtype

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

Вызов `criterion(prediction, target)` возвращает тензор потерь. Для скалярного результата вызываем `loss.backward()`. Функция потерь наследует методы `Module`, поэтому `criterion.to(device)` переносит её буферы, например веса классов, если они заданы. `criterion.state_dict()` показывает сохранённое состояние.

Оптимизатор получает параметры модели. `optimizer.step()` обновляет их по `.grad`, `optimizer.zero_grad()` очищает градиенты. Их можно вызвать в разных местах: для накопления делаем несколько `backward()`, затем один `step()`. `param_groups` хранит группы с настройками, а `state_dict()` и `load_state_dict()` позволяют сохранить и восстановить состояние, например накопленные моменты Adam.

In [ ]:
criterion.state_dict(), optimizer.param_groups[0]["lr"]

Включим режим обучения слоёв.

In [ ]:
model.train();

Очистим производные предыдущего шага. `set_to_none=True` убирает сохранённый градиент.

In [ ]:
optimizer.zero_grad(set_to_none=True)
print(model.linear.weight.grad)

Прямой проход: две оценки для каждого из шести объектов.

In [ ]:
logits = model(train_x)
logits

Функция потерь сводит весь батч к одному числу.

In [ ]:
loss = criterion(logits, train_y)
loss, loss.shape

Обратный проход вычисляет производные loss по параметрам. Что именно он обходит, разберём в следующем разделе.

In [ ]:
loss.backward()
model.linear.weight.grad

Оптимизатор изменяет параметры по градиентам. `detach().clone()` сохраняет отдельную копию прежних весов без графа.

In [ ]:
weight_before = model.linear.weight.detach().clone()
optimizer.step()
torch.equal(weight_before, model.linear.weight)

`backward()` вычисляет градиенты, `step()` меняет веса. Это два разных действия. Для следующего шага заново вычисляют `logits` и `loss`; старый loss после `step()` сам не обновится.

## 10. Что такое граф вычислений и где он находится

Граф - значения и операции, от которых зависит результат. Возьмём

$$y=xw+b,\qquad r=y-8,\qquad L=r^2.$$

При `x=2`, `w=3`, `b=1` получаем `y=7`, `r=-1`, `L=1`.

В обычном eager-режиме операции PyTorch выполняются при вызове: умножение сразу получает значение, а не ждёт `backward()`. Если запись градиентов включена и хотя бы одному входу нужна производная, autograd одновременно сохраняет связи для обратного прохода.

In [ ]:
x = torch.tensor(2., requires_grad=True)
w = torch.tensor(3., requires_grad=True)
b = torch.tensor(1., requires_grad=True)

Результат существует сразу после прямого прохода.

In [ ]:
product = x * w
y = product + b
residual = y - 8
objective = residual.square()
y, residual, objective

`grad_fn` связывает результат с узлом обратного прохода. У созданного нами листового `x` такого узла нет.

In [ ]:
x.is_leaf, x.grad_fn, y.is_leaf, type(y.grad_fn).__name__

По умолчанию `.grad` сохраняется у листовых тензоров. Попросим сохранить производные и для остальных узлов нашего примера.

In [ ]:
product.retain_grad()
y.retain_grad()
residual.retain_grad()
objective.retain_grad()
print(x.grad)

На рисунке показаны реальные узлы `grad_fn` и связи `next_functions` этого примера. `MulBackward0` умеет дифференцировать умножение, `AddBackward0` - сложение. Узел `AccumulateGrad` записывает производную в `.grad` листового тензора.

Константа 8 не требует производной и не имеет отдельного узла накопления градиента. В графе она участвует через операцию вычитания.

In [ ]:
from support.pytorch_visuals import plot_autograd
graph_tensors = {'x': x, 'w': w, 'b': b, 'x*w': product, 'y': y, 'r': residual, 'L': objective}
plot_autograd(objective, graph_tensors);

Для обратного прохода применяют правило цепочки:

$$\frac{\partial L}{\partial r}=2r=-2,\qquad
\frac{\partial r}{\partial y}=1,\qquad
\frac{\partial L}{\partial y}=-2.$$

Поскольку `y=xw+b`, имеем

$$\frac{\partial L}{\partial w}=(-2)x=-4,\qquad
\frac{\partial L}{\partial b}=-2,\qquad
\frac{\partial L}{\partial x}=(-2)w=-6.$$

Один вызов `backward()` вычислит эти производные.

In [ ]:
objective.backward()
x.grad, w.grad, b.grad, y.grad, residual.grad

In [ ]:
plot_autograd(objective, graph_tensors, backward=True);

### Прямой и обратный проход по шагам

Зелёные числа получены при вычислении, красные прочитаны из `.grad` после `backward()`. Пройдите от входов к потере, затем от потери к входам. Для этого примера $L=(xw+b-8)^2$.

In [ ]:
from support.graph_slides import animate_graph

nodes = {
    "x": (0, 1, "$x$", "input"),
    "w": (0, -1, "$w$", "input"),
    "mul": (2.5, 0, "*", "op"),
    "b": (2.5, 2.45, "$b$", "input"),
    "add": (5, 0, "+", "op"),
    "sub": (7.5, 0, "-8", "op"),
    "sq": (10, 0, "$r^2$", "op"),
    "end": (12.5, 0, "", "input"),
}

In [ ]:
edge_tensors = [
    ("x", "mul", x, "x"), ("w", "mul", w, "w"),
    ("mul", "add", product, "u"), ("b", "add", b, "b"),
    ("add", "sub", y, "y"), ("sub", "sq", residual, "r"),
    ("sq", "end", objective, "L"),
]
edges = [(start, end, value.item(), value.grad.item(), name)
         for start, end, value, name in edge_tensors]

In [ ]:
graph_widget = animate_graph(nodes, edges, title="L = (xw + b - 8)^2", size=(12, 3.4))

**Где всё это хранится и выполняется.** Связи autograd и управляющая логика находятся в процессе программы. Сами тензоры и сохранённые для backward значения находятся на своих устройствах. Здесь это CPU; для тензоров CUDA или MPS соответствующие поддерживаемые операции выполняются на ускорителе.

В примере мы запросили производную даже по входу x. При обычном обучении производные нужны прежде всего параметрам сети; входные данные обычно имеют `requires_grad=False`.

Новый прямой проход строит новую историю операций. После обычного `backward()` сохранённые для него промежуточные данные освобождаются; в наших переменных `x`, `y` и `objective` остаются их численные значения. Повторно вызывать backward на том же вычислении обычно нельзя без сохранения графа. Для обучения проще и правильно заново вычислить loss.

Вычислим новый loss. Производные в `.grad` накапливаются, если их не очищать.

In [ ]:
second_objective = (x * w + b - 8).square()
second_objective.backward()
w.grad

Получилось -8: прежние -4 плюс новые -4. Именно поэтому перед шагом обучения вызывают `optimizer.zero_grad()`.

### detach и no_grad

`detach()` возвращает тензор с теми же данными без ребра к прежнему графу. Так можно сохранить loss для лога или использовать выход одной модели как фиксированную цель другой. Изменение данных через отсоединённый тензор меняет общую память. Для независимой копии используем `detach().clone()`.

Операции оценки можно сразу вычислить под `torch.no_grad()`. Тогда новый граф для них не строится, и результат дополнительных отсоединений не требует. Для значения без `requires_grad` или для одиночного `loss.item()` отсоединение также избыточно. Если через значение надо обучать предыдущие слои, сохраняем связь с графом.

In [ ]:
y.detach().requires_grad, y.detach().clone()

Для передачи результата в NumPy используют `tensor.detach().cpu().numpy()`. Здесь три отдельных действия: отсоединить от autograd, перенести на CPU, получить NumPy-массив. Этот путь подходит для отчёта и визуализации, но не для вычислений, через которые должны проходить производные.

In [ ]:
y.detach().cpu().numpy()

## 11. train, eval и отключение градиентов

`model.train()` и `model.eval()` управляют поведением слоёв. `eval()` не выключает autograd. Для вычислений без записи производных используют `torch.no_grad()`; для чистого инференса также подходит `torch.inference_mode()`.

Проверим это на нашей модели.

In [ ]:
model.eval();
model(train_x).requires_grad

no_grad отключает запись производных для новых операций внутри контекста.

In [ ]:
with torch.no_grad():
    output_without_graph = model(train_x)
output_without_graph.requires_grad

In [ ]:
with torch.inference_mode():
    probabilities = model(train_x).softmax(dim=1)
probabilities

По строке вероятности классов суммируются в 1. `argmax(dim=1)` выбирает класс каждого объекта.

In [ ]:
probabilities.sum(dim=1), probabilities.argmax(dim=1)

### Dropout и BatchNorm

У `Dropout(p=0.5)` на train каждый элемент с вероятностью 0.5 обнуляется, остальные умножаются на 2. На eval возвращается вход без маски и дополнительного масштабирования.

In [ ]:
dropout = nn.Dropout(p=0.5)
dropout(torch.ones(8))

In [ ]:
dropout.eval();
dropout(torch.ones(8))

BatchNorm на train использует статистики текущего батча и обновляет накопленные. На eval по умолчанию использует накопленные статистики. Это состояние слоя хранится в буферах, а не в обучаемых параметрах gamma и beta.

In [ ]:
normalization = nn.BatchNorm1d(2)
normalization(features).detach().mean(dim=0)

In [ ]:
normalization.running_mean, normalization.running_var

In [ ]:
normalization.eval();
normalization(features).detach().mean(dim=0)

Нулевое среднее на train относится к нормированным значениям до обучаемого сдвига; здесь beta=0. На eval среднее текущего батча уже не обязано быть нулевым.

| Действие | Поведение Dropout/BatchNorm | Запись производных |
|:--|:--|:--|
| `model.train()` | режим обучения | само по себе не переключает |
| `model.eval()` | режим оценки | само по себе не переключает |
| `torch.no_grad()` | само по себе не переключает | выключает внутри контекста |
| `torch.inference_mode()` | само по себе не переключает | выключает; дополнительные ограничения на последующее использование тензоров в autograd |

На оценке обычно нужны оба действия: `model.eval()` и контекст без градиентов.

## 12. Собираем знакомые строки в цикл

Теперь используем DataLoader. На каждой итерации получаем батч, очищаем градиенты, вычисляем loss, делаем backward и step. Здесь всего шесть точек; 20 эпох выполняются быстро.

### Полоса прогресса tqdm

`tqdm` оборачивает итерируемый объект и показывает, сколько шагов выполнено, сколько осталось и с какой скоростью они идут. `tqdm.auto` выбирает оформление для ноутбука или терминала. Объекты внутри итератора остаются теми же.

`desc` задаёт подпись, `leave=False` убирает полосу после завершения. В работающем ноутбуке она видна во время прохода.

In [ ]:
from tqdm.auto import tqdm

In [ ]:
list(tqdm(loader, desc="Батчи", leave=False))

Теперь обернём эпохи. Через `set_postfix` покажем loss последнего батча; это не средний loss всей эпохи. На ускорителе вызов `.item()` ждёт получения числа на CPU, поэтому в большом обучении показатели обычно обновляют реже. [Документация tqdm](https://tqdm.github.io/docs/tqdm/).

In [ ]:
model.train()
progress = tqdm(range(20), desc='Эпохи', leave=False)
for epoch in progress:
    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        batch_loss = criterion(model(inputs), labels)
        batch_loss.backward()
        optimizer.step()
    progress.set_postfix(loss=f"{batch_loss.item():.3f}")

In [ ]:
model.eval()
with torch.inference_mode():
    predictions = model(train_x).argmax(dim=1)
predictions, train_y, (predictions == train_y).float().mean().item()

Это accuracy на тех же шести точках, на которых обучались. Она проверяет работу маленького примера, но ничего не доказывает о качестве на новых данных. В реальной задаче обучение, подбор параметров и итоговую оценку разделяют.

### Сохраняем веса и загружаем модель

`state_dict()` содержит параметры и зарегистрированные буферы. Для восстановления создают такую же архитектуру и загружают в неё состояние. Используем временную папку, чтобы пример не оставлял файлы рядом с ноутбуком.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
checkpoint_folder = TemporaryDirectory()
checkpoint_path = Path(checkpoint_folder.name) / 'model.pt'
torch.save(model.state_dict(), checkpoint_path)

In [ ]:
weights = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
weights.keys()

In [ ]:
restored_model = TinyClassifier()
restored_model.load_state_dict(weights)

In [ ]:
restored_model.eval()
with torch.inference_mode():
    restored_predictions = restored_model(features).argmax(dim=1)
torch.equal(restored_predictions, predictions.cpu())

## 13. Динамический граф и заранее построенный граф

В eager-режиме Python выполняет операции по мере вызова. В PyTorch autograd записывает операции **этого** прямого прохода. Если условие в Python выбрало другую ветку, следующий проход построит другие связи для backward. Посмотрим на два вызова одной функции.

In [ ]:
def piecewise(value):
    if value.item() > 0:
        result = value.square()
    else:
        result = -value
    return result

In [ ]:
positive = piecewise(torch.tensor(2., requires_grad=True))
negative = piecewise(torch.tensor(-2., requires_grad=True))
type(positive.grad_fn).__name__, type(negative.grad_fn).__name__

Граф можно построить заранее и исполнять для новых входов. Его узлы описывают операции, рёбра передают тензоры. Например, `torch.fx.symbolic_trace` запишет функцию с умножением и сложением в виде графа. У него вход `value` и два узла операций. Вызовы с другими числами используют ту же последовательность узлов.

In [ ]:
def quadratic(value):
    result = value * value + 1
    return result

In [ ]:
traced_quadratic = torch.fx.symbolic_trace(quadratic)
print(traced_quadratic.graph)

In [ ]:
traced_quadratic(torch.tensor(2.)), traced_quadratic(torch.tensor(3.))

### Как это устроено в трёх библиотеках

| | PyTorch | TensorFlow 2 | JAX |
|:--|:--|:--|:--|
| Обычные вызовы | Eager, Python сразу получает тензор | Eager, Python сразу получает тензор | Операции над `jax.Array` |
| Производная | `loss.backward()` добавляет градиенты в `.grad` | `GradientTape.gradient(loss, variables)` возвращает градиенты | `jax.grad(f)` создаёт функцию, возвращающую градиент |
| Описание модели | `nn.Module` держит параметры и буферы | `keras.Model`, изменяемые параметры `tf.Variable` | Обычно функция получает массивы параметров явно, библиотеки дают свои модули |
| Переход к исполнению графа | `torch.compile` захватывает участки программы. Между ними возможен возврат в Python | `tf.function` трассирует операции в `tf.Graph`, затем использует граф для совместимых входов | `jax.jit` трассирует и компилирует функцию. Для ветвления по значению используют `jax.lax.cond` |
| Ветвление | В eager обычный Python `if` может выбрать ветку по числу из тензора | В графе условие по тензору можно выразить через `tf.cond` | При трассировке выбор ветки по значению тензора задают операцией графа |

Исторически TensorFlow 1 часто показывали как два шага. Сначала собирали граф с заполнителями входов, затем передавали данные в `Session.run`. Так построено сравнение в курсе Stanford CS231n 2018 года. В TensorFlow 2 обычный вызов выполняется в eager. Декоратор `@tf.function` даёт отдельный этап построения графа. Смена типа, формы входа или некоторых аргументов Python может запустить повторную трассировку.

[Stanford CS231n о вычислительных графах](https://cs231n.stanford.edu/slides/2018/cs231n_2018_lecture08.pdf). [TensorFlow о графах](https://www.tensorflow.org/guide/intro_to_graphs). [PyTorch о torch.compile](https://docs.pytorch.org/tutorials/intermediate/torch_compile_tutorial.html). [JAX о JIT](https://docs.jax.dev/en/latest/201/jit.html).

### Одно вычисление в TensorFlow 2

Запись ниже показывает разницу между вызовом eager и трассировкой. Для семинара достаточно выполнить эквивалентный пример на PyTorch выше.

```python
import tensorflow as tf

def quadratic(value):
    result = value * value + 1
    return result

quadratic(tf.constant(2.))             # eager, результат 5
graph_quadratic = tf.function(quadratic)
graph_quadratic(tf.constant(2.))       # первая трассировка и запуск, результат 5
graph_quadratic(tf.constant(3.))       # тот же граф, результат 10
```

В этом примере меняется только значение скаляра. Граф можно переиспользовать. Узлы сложения и умножения выполняет рантайм на выбранном устройстве.

### Справка к примерам

- [Тензоры и NumPy](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html).
- [Dataset и DataLoader](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html).
- [Создание модели](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html).
- [Autograd: вычислительный граф](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html).
- [Autograd: сохранённые значения и освобождение графа](https://docs.pytorch.org/docs/stable/notes/autograd.html).

## 14. Как градиент накапливается за четыре микробатча

Хотим посчитать градиент для четырёх объектов и сделать **одно** обновление. Вместо одного большого батча пропускаем объекты по одному. У всех микробатчей одинаковый размер, поэтому вклад каждого равен градиенту его средней потери, делённой на 4.

$$
\begin{aligned}
L &= \frac{L_1+L_2+L_3+L_4}{4},\\
\frac{\partial L}{\partial w} &= \sum_{i=1}^4 \frac{\partial (L_i/4)}{\partial w}.
\end{aligned}
$$

В начале группы очищаем `w.grad`. Каждый `backward()` добавляет свою производную в этот буфер. Вес остаётся прежним до `step()`. Если размеры микробатчей разные, их средние умножают на долю объектов в группе. При BatchNorm статистики считаются на каждом микробатче, поэтому результат может отличаться от одного большого батча.

In [ ]:
accum_w = nn.Parameter(torch.tensor(0.))
accum_optimizer = torch.optim.SGD([accum_w], lr=0.1)
accum_x = torch.arange(1., 5.)
accum_target = 2 * accum_x

In [ ]:
accum_optimizer.zero_grad(set_to_none=True)
accum_w.grad

In [ ]:
grad_after_microbatch = []
micro_count = len(accum_x)
for input_value, target_value in zip(accum_x, accum_target):
    prediction = accum_w * input_value
    scaled_loss = (prediction - target_value).square() / micro_count
    scaled_loss.backward()
    grad_after_microbatch.append(accum_w.grad.item())

grad_after_microbatch

In [ ]:
batch_loss_check = (accum_w * accum_x - accum_target).square().mean()
batch_gradient, = torch.autograd.grad(batch_loss_check, accum_w)
accum_w.grad, batch_gradient

In [ ]:
accum_optimizer.step()
accum_w.item()

На каждом проходе строится граф для текущего микробатча. Градиенты накапливаются в одном буфере `w.grad`. После четырёх проходов в буфере −30, как и при одном проходе по полному батчу. Один шаг оптимизатора меняет вес с 0 до 3: $0-0.1\cdot(-30)=3$.